# HAR Stage 7 — GSE270828 metadata QC, **NO Google Drive**

**Run in Google Colab with a Python runtime.** This notebook **does not mount, connect to, or request permissions for Google Drive**. Data downloads directly from the NCBI GEO public HTTPS server into Colab's temporary `/content/` storage. Results are exported as a small ZIP to your computer.

**Two levels of analysis:**
1. **Fast, small-file audit:** Original guide annotations and GEO sample metadata; no big files or special setup.
2. **Optional full metadata audit:** Download the original **2.31 GB** compressed `.rds.gz` and ask R to extract cell metadata without calculating differential expression. This can be slow or run out of RAM on free Colab. It is enabled in the settings below; switch it off if the large file fails.

**Scientific caveats:** sequencing lanes are not independent donors; guide-design gene labels are not experimentally validated targets; results here are **descriptive QC only** — no DEG tests, invented P values, or claims of HAR→gene causality.

Sources: [NCBI GSE270828](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE270828) · [NoonanLab author code](https://github.com/NoonanLab/Noble_et_al_2024).

In [ ]:
# 1. Runtime dependencies (no account authentication)
%pip -q install "pandas>=2.0,<3.0" "requests>=2.28,<3.0" "psutil>=5.9,<8.0"
import json, os, shutil, subprocess, traceback
from pathlib import Path
import pandas as pd
print('Current working directory:', Path.cwd())
print('No Google Drive connection is requested.')

## 2. Load validated QC functions

This cell includes all required Python functions, so you do **not** need a separate `.py` file or a Drive folder.

In [ ]:
"""GSE270828 Colab preflight, direct NCBI delivery, metadata-only QC.

No Google Drive account, mount, token, or API is involved. All analyses
are descriptive. No differential expression or causal inference is run.
"""
from __future__ import annotations

import csv
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

GEO_BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828"
GUIDE_URL = GEO_BASE + "/suppl/GSE270828_feature_README.csv"
MATRIX_URL = GEO_BASE + "/matrix/GSE270828_series_matrix.txt.gz"
RDS_URL = GEO_BASE + "/suppl/GSE270828_Merged_CountMatrix.rds.gz"
EXPECTED_RDS_SIZE_BYTES = 2312020968  # observed original file inventory, recheck via HEAD
CONTROL_RE = re.compile(r"(?:^ntc(?:\d|_|\b)|non[-_ ]?target|negative[-_ ]?control|^cont[_-]|^ctrl\b|scrambl)", re.I)


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def checksum_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def download_small(url, dest, timeout=45):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, timeout=(20, timeout), stream=True, headers={'User-Agent':'HAR-Stage7-Research/1.0'}) as r:
        r.raise_for_status()
        with dest.open('wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)
    if dest.stat().st_size < 100:
        raise RuntimeError(f"Unexpected tiny file: {dest}")
    return dest


def audit_guide(guide_csv, out_dir):
    out = Path(out_dir)
    out.mkdir(exist_ok=True, parents=True)
    df = pd.read_csv(guide_csv, dtype='string')
    required = {'id', 'name', 'sequence', 'feature_type', 'target_gene_name'}
    if not required.issubset(df.columns):
        raise ValueError('Unexpected guide file columns: ' + ', '.join(map(str, df.columns)))
    df['control_guide'] = df['id'].str.match(r'^(?:NTC\d+|NTC_10x|cont[_-])', case=False, na=False)
    df['HAR_screened'] = ~df['control_guide']
    har = df.loc[df['HAR_screened']].groupby('name', dropna=False).agg(
        n_guides=('id','size'), distinct_sequences=('sequence','nunique'),
        design_target_names=('target_gene_name', lambda x:'; '.join(sorted(x.dropna().unique()))),
    ).reset_index().rename(columns={'name':'HAR'})
    har.to_csv(out/'design_HAR_guide_counts.tsv', sep='\t', index=False)
    df.loc[df['control_guide']].to_csv(out/'guide_controls.tsv', sep='\t', index=False)
    (out/'screened_HAR_labels.txt').write_text('\n'.join(sorted(har['HAR'].astype(str)))+'\n',encoding='utf-8')
    summary = {
        'source_url': GUIDE_URL,
        'guide_rows_total': len(df),
        'designed_HARs': len(har),
        'HAR_guides': int(df['HAR_screened'].sum()),
        'control_guides_total': int(df['control_guide'].sum()),
        'non_targeting_guide_count': int(df['id'].str.match(r'^NTC(?:\d+|_10x)', case=False, na=False).sum()),
        'design_annotations_are_not_validated_gene_targets': True,
        'sha256_original_guide': checksum_sha256(guide_csv),
    }
    write_json(out/'guide_design_qc.json', summary)
    return summary, sorted(har['HAR'].astype(str).tolist())


def audit_geo_series_matrix(matrix_gz, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True,exist_ok=True)
    fields = {}
    with gzip.open(matrix_gz, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!Sample_'):
                row = next(csv.reader([line.rstrip('\r\n')], delimiter='\t'))
                fields[row[0]] = row[1:]
    if not fields:
        raise ValueError('No !Sample_* metadata in GEO series matrix')
    sizes = Counter(len(x) for x in fields.values())
    if len(sizes)>1:
        raise ValueError(f'GEO sample metadata row sizes differ: {sizes}')
    sample_count = next(iter(sizes))
    if sample_count > 0:
        dat = pd.DataFrame({col:values for col,values in fields.items()})
        dat.insert(0,'GEO_metadata_index',range(1,len(dat)+1))
        dat.to_csv(out/'GEO_sample_metadata.tsv',sep='\t',index=False)
    summary = {'GEO_sample_metadata_entries':int(sample_count),'GEO_sample_fields':list(fields),
               'sample_rows_are_not_independent_donors':True,
               'sha256_original_matrix':checksum_sha256(matrix_gz)}
    write_json(out/'geo_sample_metadata_qc.json', summary)
    return summary


def get_cgroup_memory_bytes():
    for p in ('/sys/fs/cgroup/memory.max','/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        try:
            s=Path(p).read_text().strip()
            if s!='max':
                n=int(s)
                if 0<n<2**50:return n
        except (OSError,ValueError):pass
    try:
        import psutil
        return int(psutil.virtual_memory().total)
    except Exception:
        return None


def head_remote_size(url):
    try:
        with requests.head(url, allow_redirects=True, timeout=(15,30)) as r:
            r.raise_for_status()
            return int(r.headers['Content-Length']) if r.headers.get('Content-Length','').isdigit() else None
    except requests.RequestException:
        return None


def download_large_resumable(url, dest, min_free_extra=3*1024**3):
    """Stream file to /content, resume only if server honors byte ranges.

    Full-size check is necessary; this function does NOT parse or authenticate the RDS.
    """
    dest=Path(dest);dest.parent.mkdir(parents=True, exist_ok=True)
    expected=head_remote_size(url) or EXPECTED_RDS_SIZE_BYTES
    free=shutil.disk_usage(dest.parent).free
    existing=dest.stat().st_size if dest.exists() else 0
    if existing==expected:
        return {'status':'already_downloaded','bytes':existing,'remote_expected':expected,'source':url}
    partial=dest.with_name(dest.name+'.part')
    size=partial.stat().st_size if partial.exists() else 0
    needed=max(0,expected-size)
    if free<needed+min_free_extra:
        raise RuntimeError(f'Insufficient free local storage: need about {(needed+min_free_extra)/1e9:.1f} GB, have {free/1e9:.1f} GB.')
    headers={'User-Agent':'HAR-Stage7-Research/1.0'}
    if size: headers['Range']=f'bytes={size}-'
    with requests.get(url,headers=headers,stream=True,timeout=(30,120)) as r:
        r.raise_for_status()
        if size and r.status_code==206:
            content_range=r.headers.get('Content-Range','')
            if not content_range.startswith(f'bytes {size}-'):
                raise RuntimeError(f'Resumed response is not aligned with partial file: {content_range}')
            mode='ab'
        elif size and r.status_code==200:
            size=0;mode='wb'
        else:
            mode='wb'
        with partial.open(mode) as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk:f.write(chunk)
    actual=partial.stat().st_size
    if expected and actual!=expected:
        raise RuntimeError(f'Incomplete original GEO RDS: got {actual:,} bytes, expected {expected:,}. Re-run cell to resume.')
    partial.replace(dest)
    return {'status':'downloaded','bytes':actual,'remote_expected':expected,'source':url}


def audit_cell_metadata(meta_tsv, out_dir, screened_hars, source='GSE270828 original RDS'):
    """Metadata-only, donor-aware cautions. No DE analysis."""
    out=Path(out_dir); out.mkdir(parents=True,exist_ok=True)
    obs=pd.read_csv(meta_tsv,sep='\t',low_memory=False)
    for c in ('har','guide','replicate','rep','lane','batch','perturbation_name'):
        if c in obs: obs[c]=obs[c].astype('string').fillna('<missing>').str.strip()
    schema=pd.DataFrame([{
        'field':k,'dtype':str(obs[k].dtype),'missing_count':int(obs[k].isna().sum()),
        'distinct_nonmissing_values':int(obs[k].nunique(dropna=True))} for k in obs.columns])
    schema.to_csv(out/'metadata_schema.tsv',sep='\t',index=False)
    result={'source':source,'n_cells_metadata_rows':len(obs),'metadata_columns':list(obs.columns),
            'has_har_label':'har' in obs,'has_guide_label':'guide' in obs,
            'has_biological_donor_identifiers':False,'donor_identity_not_assumed_from_replicate_or_lane':True,
            'no_statistical_effects_computed':True}
    if 'har' not in obs:
        result['qc_status']='missing HAR assignment column; no HAR coverage calculated'
        write_json(out/'single_cell_metadata_summary.json',result)
        return result
    hc=obs.groupby('har',dropna=False).size().rename('n_cells').reset_index().sort_values('n_cells',ascending=False)
    hc['possible_control_label_not_confirmed']=hc['har'].map(lambda x:bool(CONTROL_RE.search(str(x))))
    hc.to_csv(out/'har_cell_counts.tsv',sep='\t',index=False)
    controls=hc.loc[hc['possible_control_label_not_confirmed']]
    controls.to_csv(out/'possible_control_har_labels.tsv',sep='\t',index=False)
    labels=set(map(str,screened_hars))
    cover=pd.DataFrame({'HAR':sorted(labels)}).merge(hc[['har','n_cells']],how='left',left_on='HAR',right_on='har').drop(columns=['har'])
    cover['n_cells']=cover['n_cells'].fillna(0).astype('int64')
    cover['observed_in_metadata']=cover['n_cells']>0
    replicate_col=next((c for c in ['replicate','rep'] if c in obs),None)
    for col,filename in [('guide','har_by_guide.tsv'),(replicate_col,'har_by_replicate_label.tsv'),('lane','har_by_lane.tsv'),('batch','har_by_batch.tsv')]:
        if col and col in obs.columns:
            tab=obs.groupby(['har',col],dropna=False).size().rename('n_cells').reset_index()
            tab.to_csv(out/filename,sep='\t',index=False)
            cnt=obs.groupby('har')[col].nunique().rename('n_distinct_'+col+'_labels')
            cover=cover.join(cnt,on='HAR')
            cover['n_distinct_'+col+'_labels']=cover['n_distinct_'+col+'_labels'].fillna(0).astype('int64')
    cover.to_csv(out/'screened_HAR_coverage.tsv',sep='\t',index=False)
    result.update({'qc_status':'metadata successfully audited',
                   'n_HARs_in_original_design':len(labels),
                   'n_HARs_observed_in_metadata':int(cover['observed_in_metadata'].sum()),
                   'n_unique_har_labels_in_metadata':int(hc.shape[0]),
                   'n_possible_control_har_labels_heuristic_only':int(len(controls)),
                   'replicate_label_col':replicate_col,
                   'n_unique_replicate_labels':int(obs[replicate_col].nunique()) if replicate_col else None,
                   'n_unique_lane_labels':int(obs['lane'].nunique()) if 'lane' in obs else None})
    write_json(out/'single_cell_metadata_summary.json',result)
    return result


## 3. Settings and local output directory

Default is to try the original NCBI **2.31 GB** RDS. Set `DOWNLOAD_LARGE_RDS = False` to run the small metadata-only preflight without that download. An existing original file in `/content` is reused. No `.h5ad` conversion is required.

In [ ]:
DOWNLOAD_LARGE_RDS = True  # @param {type:"boolean"}
EXTRACT_CELL_METADATA = True  # @param {type:"boolean"}
MIN_AVAILABLE_RAM_GB = 9.5  # @param {type:"number"}

BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
DATA = BASE / 'GSE270828_public_source'
OUT = BASE / 'HAR_Stage7_NoDrive_Results'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
STATUS = {'download_method':'NCBI GEO public HTTPS; no Google Drive', 'stage':'Stage 7',
          'differential_expression_performed':False,'biological_donors_inferred':False,
          'full_download_requested':DOWNLOAD_LARGE_RDS,'full_metadata_extraction_requested':EXTRACT_CELL_METADATA,
          'source_urls':{'guide':GUIDE_URL,'geo_metadata':MATRIX_URL,'original_counts_RDS':RDS_URL}}
print('Data folder:',DATA)
print('QC output folder:',OUT)
print('Large input requested:',DOWNLOAD_LARGE_RDS)
print('Available disk GB:',round(shutil.disk_usage(DATA).free/1e9,2))
mem=get_cgroup_memory_bytes()
print('Memory limit GB:', round(mem/1e9,2) if mem else 'unknown')

## 4. Download two small original GEO files (automatic)

These are official GEO files, not computationally generated counts. If the server is temporarily unavailable, that failure will be recorded; the results ZIP will still be generated.

In [ ]:
downloads = {}
for label, url, dest in [
    ('guide_reference', GUIDE_URL, DATA/'GSE270828_feature_README.csv'),
    ('geo_sample_metadata', MATRIX_URL, DATA/'GSE270828_series_matrix.txt.gz')]:
    try:
        if not dest.exists() or dest.stat().st_size < 100:
            download_small(url,dest)
        downloads[label]={'status':'ok','local_path':str(dest),'size_bytes':dest.stat().st_size}
        print(label,'OK',dest.stat().st_size,'bytes')
    except Exception as exc:
        downloads[label]={'status':'error','message':str(exc)}
        print(label,'FAILED:',str(exc))
STATUS['small_file_downloads']=downloads
write_json(OUT/'STATUS.json',STATUS)

## 5. Verify original guide design and sample metadata

Counts are computed **only** from downloaded source files. Non-targeting controls and other control guides are reported separately.

In [ ]:
screen_labels=[]
try:
    guide_file=DATA/'GSE270828_feature_README.csv'
    if guide_file.exists():
        guide_summary,screen_labels=audit_guide(guide_file,OUT)
        STATUS['guide_design_qc']=guide_summary
        print('Guide verification:',json.dumps(guide_summary,indent=2))
    else:
        STATUS['guide_design_error']='Original guide CSV missing'
except Exception as exc:
    STATUS['guide_design_error']=str(exc)
    print('Guide QC error:',exc)
try:
    matrix_file=DATA/'GSE270828_series_matrix.txt.gz'
    if matrix_file.exists():
        STATUS['geo_sample_metadata_qc']=audit_geo_series_matrix(matrix_file,OUT)
        print('GEO metadata audit:', STATUS['geo_sample_metadata_qc']['GEO_sample_metadata_entries'],'entries')
    else:
        STATUS['geo_metadata_error']='GEO Series Matrix missing'
except Exception as exc:
    STATUS['geo_metadata_error']=str(exc)
    print('GEO metadata error:',exc)
write_json(OUT/'STATUS.json',STATUS)

## 6. Download original large RDS **directly from NCBI**

**Please read:** This step can transfer approximately 2.31 GB and may need several GB of extra disk space. It does not use any Google Drive account. If Colab disconnects, the temporary file can be lost. If storage is insufficient or the network blocks NCBI, the status will be saved and metadata inference will be skipped.

In [ ]:
rds_file=DATA/'GSE270828_Merged_CountMatrix.rds.gz'
if DOWNLOAD_LARGE_RDS:
    try:
        info=download_large_resumable(RDS_URL,rds_file)
        STATUS['large_RDS_download']=info
        print('Original RDS:',info)
    except Exception as exc:
        STATUS['large_RDS_download']={'status':'error','error':str(exc)}
        print('Original RDS unavailable:',exc)
else:
    STATUS['large_RDS_download']={'status':'skipped_by_user','explanation':'small-file-only QC'}
write_json(OUT/'STATUS.json',STATUS)

## 7. Optionally extract cell metadata from `.rds.gz` using R

An RDS is **not** an `.h5ad`. This step uses `Rscript` to read the original object and export only its `meta.data` slot (if available). On a constrained Colab runtime, full object deserialization can exceed available memory; this is detected where possible and failures are recorded. The notebook never claims success unless the output file is actually created.

In [ ]:
R_SCRIPT = r'''args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(methods))
if (length(args) < 3) stop("Expected source.RDS.gz output.tsv status.txt")
f <- args[[1]]; output <- args[[2]]; status <- args[[3]]
writeLines("started", status)
con <- gzfile(f, open="rb")
obj <- readRDS(con)
close(con)
writeLines("RDS deserialized", status)
if (isS4(obj)) {
  sn <- slotNames(obj)
  if (!("meta.data" %in% sn)) stop(paste("RDS has no meta.data slot. Slots:",paste(sn,collapse=",")))
  md <- as.data.frame(slot(obj,"meta.data"))
} else if (is.list(obj) && !is.null(obj$meta.data)) {
  md <- as.data.frame(obj$meta.data)
} else {
  stop(paste("Unsupported original object class:",paste(class(obj),collapse=",")))
}
md$cell_barcode <- rownames(md)
write.table(md, file=output, sep="\t", quote=TRUE, row.names=FALSE, na="")
writeLines(paste("success_rows",nrow(md),"columns",ncol(md)), status)
cat("Metadata extracted:",nrow(md),"rows,",ncol(md),"columns\n")
'''
R_FILE=BASE/'stage7_extract_cell_metadata.R'
R_FILE.write_text(R_SCRIPT,encoding='utf-8')
meta_file=DATA/'GSE270828_original_cell_metadata.tsv'
try:
    if not EXTRACT_CELL_METADATA:
        STATUS['RDS_metadata_extraction']={'status':'skipped_by_user'}
    elif not rds_file.is_file():
        STATUS['RDS_metadata_extraction']={'status':'skipped','reason':'Original 2.31-GB RDS not available'}
    else:
        mem=get_cgroup_memory_bytes()
        if mem and mem/1e9 < MIN_AVAILABLE_RAM_GB:
            raise RuntimeError(f'Available RAM limit {mem/1e9:.2f} GB below safety threshold {MIN_AVAILABLE_RAM_GB:.2f} GB; switch to high-memory runtime or keep small-file QC.')
        if not shutil.which('Rscript'):
            print('Installing base R (required to read RDS; not installing full Seurat package)...')
            inst=subprocess.run(['bash','-lc','apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base-core'],
                                capture_output=True,text=True,timeout=1200)
            (OUT/'R_install_last_lines.txt').write_text((inst.stdout+'\n'+inst.stderr)[-5000:])
            if inst.returncode != 0:
                raise RuntimeError('Could not install Rscript: '+inst.stderr[-500:])
        if not shutil.which('Rscript'):
            raise RuntimeError('Rscript still unavailable')
        print('Reading compressed original R object; this can be memory-intensive.')
        # Bound R virtual memory to reduce likelihood of killing the whole notebook.
        def set_ram_limit():
            import resource
            ram=get_cgroup_memory_bytes()
            if ram:
                cap=int(ram*0.90)
                resource.setrlimit(resource.RLIMIT_AS,(cap,cap))
        proc=subprocess.run(['Rscript',str(R_FILE),str(rds_file),str(meta_file),str(OUT/'RDS_read_progress.txt')],
                            capture_output=True,text=True,timeout=1800,
                            preexec_fn=set_ram_limit)
        (OUT/'RDS_Rscript_stdout_stderr.txt').write_text((proc.stdout+'\n--- STDERR ---\n'+proc.stderr)[-12000:])
        if proc.returncode != 0 or not meta_file.is_file() or meta_file.stat().st_size<100:
            raise RuntimeError('R could not deserialize/export the full dataset (possible RAM limit, unsupported class or corrupt file). See RDS_Rscript_stdout_stderr.txt. Return code: '+str(proc.returncode))
        STATUS['RDS_metadata_extraction']={'status':'success','metadata_tsv':str(meta_file),
                                           'metadata_bytes':meta_file.stat().st_size}
        print('Metadata export succeeded:',meta_file.stat().st_size,'bytes')
except Exception as exc:
    STATUS['RDS_metadata_extraction']={'status':'error','message':str(exc)}
    print('Metadata extraction not completed:',exc)
write_json(OUT/'STATUS.json',STATUS)

## 8. Real cell metadata QC (only if RDS extraction succeeded)

The notebook audits HAR labels, cell counts, guide assignments, replicate/batch labels, control-like labels and source schema. It **never treats a sequencing lane as an independent donor**.

In [ ]:
try:
    if meta_file.exists() and meta_file.stat().st_size>100 and screen_labels:
        STATUS['cell_level_metadata_qc']=audit_cell_metadata(meta_file,OUT,screen_labels,
                                                             source='GSE270828 original NCBI GEO RDS')
        print('Cells in extracted metadata:',STATUS['cell_level_metadata_qc']['n_cells_metadata_rows'])
    else:
        STATUS['cell_level_metadata_qc']={'status':'not_run',
            'reason':'Original metadata was not extracted or guide list not available'}
        print('Original-cell QC not run: input metadata not yet available.')
except Exception as exc:
    STATUS['cell_level_metadata_qc']={'status':'error','message':str(exc)}
    print('Cell metadata QC error:',exc)
write_json(OUT/'STATUS.json',STATUS)

## 9. Download **small results ZIP** to your computer

Only QC tables and logs are included, **never** the multi-GB RDS or full individual-cell expression matrix. Upload the ZIP into this ChatGPT conversation for the next scientific step.

In [ ]:
(OUT/'README_FIRST.txt').write_text(
    'HAR Stage 7 — Direct original NCBI GEO files, no Google Drive.\n'
    'Open STATUS.json to identify completed/blocked stages.\n'
    'This is metadata QC only; gene-level perturbation expression effects have NOT been computed.\n'
    'A failed large download or failed RDS extraction does NOT imply missing biological data.\n',encoding='utf-8')
write_json(OUT/'STATUS.json',STATUS)
archive=shutil.make_archive(str(BASE/'HAR_Stage7_NoDrive_Results'),'zip',OUT)
print('QC output ZIP:',archive)
print('ZIP size:',Path(archive).stat().st_size,'bytes')
print('Summary of completed steps:')
print(json.dumps({k:v for k,v in STATUS.items() if k in ('small_file_downloads','guide_design_qc','large_RDS_download','RDS_metadata_extraction','cell_level_metadata_qc')},indent=2,default=str)[:3000])
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Colab, retrieve ZIP from:',archive)